# **SUNO BARK**

In [ ]:
import os
# ==========================================
# THE BUG BYPASS (Carried over to be safe!)
# ==========================================
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"
# ==========================================

import json
import torch
import scipy.io.wavfile
import warnings
from transformers import AutoProcessor, BarkModel

# Suppress minor generation warnings
warnings.filterwarnings("ignore")

print("Loading evaluation dataset...")
with open("hindi_evaluation_set.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# 1. Setup device and load the model
device = "cuda" if torch.cuda.is_available() else "cpu"
# Using the 'small' variant for faster inference. Change to "suno/bark" for max quality if you have high VRAM.
model_id = "suno/bark-small"

print(f"Loading {model_id} (this will download the weights)...")
processor = AutoProcessor.from_pretrained(model_id)
model = BarkModel.from_pretrained(model_id).to(device)

# 2. Prepare output directory
output_dir = "bark_hindi_eval"
os.makedirs(output_dir, exist_ok=True)

# 3. Define Bark's built-in Hindi Speaker Presets
voice_presets = {
    "Male": "v2/hi_speaker_0",
    "Female": "v2/hi_speaker_1"
}

print("\nStarting Audio Generation...")

# 4. Generation Loop
for gender, preset in voice_presets.items():
    print(f"\n--- Processing {gender} Voice ({preset}) ---")

    for key, item in eval_data.items():
        text = item["text"]
        filename = f"{output_dir}/{item['id']}_{gender}.wav"

        print(f"Synthesizing {item['id']}...")

        # Process text and speaker preset
        inputs = processor(text, voice_preset=preset, return_tensors="pt").to(device)

        # Generate audio (Bark uses pad_token_id=10000 internally)
        with torch.no_grad():
            audio_array = model.generate(**inputs, pad_token_id=10000)

        # Extract audio and save
        audio_data = audio_array.cpu().numpy().squeeze()
        sample_rate = model.generation_config.sample_rate

        scipy.io.wavfile.write(filename, rate=sample_rate, data=audio_data)

print(f"\n✅ All done! Evaluation audio files saved to '{output_dir}'.")

Loading evaluation dataset...
Loading suno/bark-small (this will download the weights)...


tokenizer_config.json:   0%|          | 0.00/353 [00:00<?, ?B/s]

speaker_embeddings_path.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Synthesizing HIN_02...
Synthesizing HIN_03...
Synthesizing HIN_04...
Synthesizing HIN_05...
Synthesizing HIN_06...
Synthesizing HIN_07...
Synthesizing HIN_08...
Synthesizing HIN_09...
Synthesizing HIN_10...
Synthesizing HIN_11...
Synthesizing HIN_12...
Synthesizing HIN_13...
Synthesizing HIN_14...
Synthesizing HIN_15...
Synthesizing HIN_16...
Synthesizing HIN_17...
Synthesizing HIN_18...
Synthesizing HIN_19...
Synthesizing HIN_20...

--- Processing Female Voice (v2/hi_speaker_1) ---
Synthesizing HIN_01...


speaker_embeddings/v2/hi_speaker_1_seman(…):   0%|          | 0.00/2.63k [00:00<?, ?B/s]

speaker_embeddings/v2/hi_speaker_1_coars(…):   0%|          | 0.00/7.65k [00:00<?, ?B/s]

speaker_embeddings/v2/hi_speaker_1_fine_(…):   0%|          | 0.00/15.2k [00:00<?, ?B/s]

Synthesizing HIN_02...
Synthesizing HIN_03...
Synthesizing HIN_04...
Synthesizing HIN_05...
Synthesizing HIN_06...
Synthesizing HIN_07...
Synthesizing HIN_08...
Synthesizing HIN_09...
Synthesizing HIN_10...
Synthesizing HIN_11...
Synthesizing HIN_12...
Synthesizing HIN_13...
Synthesizing HIN_14...
Synthesizing HIN_15...
Synthesizing HIN_16...
Synthesizing HIN_17...
Synthesizing HIN_18...
Synthesizing HIN_19...
Synthesizing HIN_20...

✅ All done! Evaluation audio files saved to 'bark_hindi_eval'.


In [ ]:
import whisper
import jiwer
import json
import os
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

print("Loading evaluation dataset...")
with open("hindi_evaluation_set.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

print("Loading Whisper 'medium' model...")
model = whisper.load_model("medium")

output_dir = "bark_hindi_eval"
results = []

print("\nStarting Transcription and Evaluation...\n")

# Evaluation Loop
for gender in ["Male", "Female"]:
    for key, item in eval_data.items():
        file_id = item["id"]
        ground_truth = item["text"]
        audio_path = f"{output_dir}/{file_id}_{gender}.wav"

        if not os.path.exists(audio_path):
            continue

        # Transcribe
        transcription_result = model.transcribe(audio_path, language="hi")
        hypothesis = transcription_result["text"].strip()

        # Calculate Error Rates
        try:
            wer = jiwer.wer(ground_truth, hypothesis)
            cer = jiwer.cer(ground_truth, hypothesis)
        except ValueError:
            wer = 1.0
            cer = 1.0

        # Store data
        results.append({
            "Test_ID": file_id,
            "Category": key,
            "Gender": gender,
            "WER": round(wer, 3),
            "CER": round(cer, 3),
            "Original_Text": ground_truth,
            "Whisper_Transcription": hypothesis
        })

        print(f"[{file_id} - {gender}] WER: {wer:.3f} | CER: {cer:.3f}")

# Create DataFrame and summarize
df_results = pd.DataFrame(results)

print("\n" + "="*40)
print("🏆 SUNO BARK EVALUATION SUMMARY")
print("="*40)

summary = df_results.groupby("Gender")[["WER", "CER"]].mean()
print(summary)

# Save report
csv_filename = "bark_whisper_evaluation.csv"
df_results.to_csv(csv_filename, index=False)
print(f"\n✅ Detailed breakdown saved to '{csv_filename}'.")

Loading evaluation dataset...
Loading Whisper 'medium' model...

Starting Transcription and Evaluation...

[HIN_01 - Male] WER: 0.467 | CER: 0.217
[HIN_02 - Male] WER: 0.722 | CER: 0.299
[HIN_03 - Male] WER: 0.750 | CER: 0.378
[HIN_04 - Male] WER: 0.545 | CER: 0.286
[HIN_05 - Male] WER: 0.692 | CER: 0.286
[HIN_06 - Male] WER: 0.583 | CER: 0.236
[HIN_07 - Male] WER: 0.417 | CER: 0.182
[HIN_08 - Male] WER: 0.714 | CER: 0.279
[HIN_09 - Male] WER: 0.786 | CER: 0.394
[HIN_10 - Male] WER: 0.500 | CER: 0.229
[HIN_11 - Male] WER: 0.857 | CER: 0.465
[HIN_12 - Male] WER: 0.615 | CER: 0.239
[HIN_13 - Male] WER: 0.231 | CER: 0.075
[HIN_14 - Male] WER: 0.786 | CER: 0.453
[HIN_15 - Male] WER: 0.533 | CER: 0.298
[HIN_16 - Male] WER: 0.588 | CER: 0.203
[HIN_17 - Male] WER: 0.786 | CER: 0.368
[HIN_18 - Male] WER: 0.643 | CER: 0.338
[HIN_19 - Male] WER: 1.000 | CER: 0.792
[HIN_20 - Male] WER: 0.533 | CER: 0.284
[HIN_01 - Female] WER: 0.467 | CER: 0.217
[HIN_02 - Female] WER: 0.611 | CER: 0.234
[HIN_03 -

In [ ]:
import shutil
from google.colab import files

print("Zipping the audio files...")
# Compress the folder into a file named 'xtts_audio_results.zip'
shutil.make_archive('suno_bark_audio_results', 'zip', 'bark_hindi_eval')

print("Starting download...")
# Trigger the browser download
files.download('suno_bark_audio_results.zip')

Zipping the audio files...
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>